In [ ]:
# Cell 1 — Setup / Sync latest repo
import os

REPO_DIR = "/content/mathphd-plus-plus"
REPO_URL = "https://github.com/Edmon02/mathphd-plus-plus.git"

if os.path.exists(REPO_DIR):
    %cd /content/mathphd-plus-plus
    !git fetch origin
    !git checkout main
    !git pull --ff-only origin main
else:
    %cd /content
    !git clone {REPO_URL}
    %cd /content/mathphd-plus-plus

%pip install -e ".[full]"

# Config flag used throughout the notebook
QUICK_TEST = True

Cloning into 'mathphd-plus-plus'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 141 (delta 68), reused 115 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 92.16 KiB | 1.28 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/mathphd-plus-plus
Obtaining file:///content/mathphd-plus-plus
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/4

In [ ]:
# Install dependencies
%pip install -q torch transformers datasets peft bitsandbytes trl accelerate sympy wandb sentencepiece einops tqdm

import os
import sys

# Mount Google Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')

# Create checkpoint directory on Drive
GDRIVE_ROOT = '/content/drive/MyDrive/mathphd_checkpoints'
os.makedirs(GDRIVE_ROOT, exist_ok=True)

# Prefer the freshly synced repo clone. Fall back to Drive copy only if needed.
REPO_PATH = '/content/mathphd-plus-plus'
DRIVE_CODE_PATH = '/content/drive/MyDrive/mathphd_plus_plus'
if os.path.exists(REPO_PATH):
    os.chdir(REPO_PATH)
    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)
    print(f'Using repo from: {REPO_PATH}')
elif os.path.exists(DRIVE_CODE_PATH):
    sys.path.insert(0, os.path.dirname(DRIVE_CODE_PATH))
    print(f'Using Drive copy from: {DRIVE_CODE_PATH}')
else:
    print('WARNING: repo code not found in /content or Drive; imports may fail.')

# Verify GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

Mounted at /content/drive
Please upload the mathphd_plus_plus folder to /content/ or Google Drive
GPU: Tesla T4
VRAM: 15.6 GB
PyTorch: 2.10.0+cu128


In [3]:
# Configuration
from mathphd_plus_plus.configs.base_config import MathPhDConfig

config = MathPhDConfig()
config.gdrive_checkpoint_root = GDRIVE_ROOT
config.use_wandb = False  # Set True if you have a wandb API key

# Reduce dataset sizes for quick testing (set to None for full training)
QUICK_TEST = True  # Set False for real training

if QUICK_TEST:
    config.cpt.target_tokens = 5_000_000  # 5M tokens instead of 500M
    config.cpt.num_train_epochs = 1
    config.cpt.save_steps = 50
    config.sft.num_train_epochs = 2
    config.prm.num_train_epochs = 1
    config.grpo.num_grpo_epochs = 1
    config.grpo.problems_per_epoch = 500  # 100 is too few for GRPO to learn

print('Config loaded. QUICK_TEST =', QUICK_TEST)

Config loaded. QUICK_TEST = True


In [4]:
from mathphd_plus_plus.models.base_model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer(
    model_name=config.model.model_name,
    special_tokens=config.model.special_tokens,
    gradient_checkpointing=config.model.gradient_checkpointing,
)

# Memory check
allocated = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory allocated: {allocated:.2f} GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: 494.0M params, 1.98 GB
GPU memory allocated: 1.98 GB


In [5]:
from mathphd_plus_plus.data.download import download_sft_data, download_prm_data, download_eval_data

CACHE_DIR = '/content/data_cache'

# Download datasets (SFT + PRM + Eval first, CPT streaming later)
print('Downloading SFT datasets...')
sft_raw = download_sft_data(cache_dir=CACHE_DIR)

print('\nDownloading PRM data...')
prm_raw = download_prm_data(cache_dir=CACHE_DIR)

print('\nDownloading eval data...')
eval_raw = download_eval_data(cache_dir=CACHE_DIR)

print('\nAll downloads complete!')

[SFT] Downloading MetaMathQA...


README.md: 0.00B [00:00, ?B/s]

MetaMathQA-395K.json:   0%|          | 0.00/396M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/395000 [00:00<?, ? examples/s]

[SFT] Downloading MATH...


...


[PRM] Downloading math-shepherd (process reward data)...


README.md: 0.00B [00:00, ?B/s]

math-shepherd.jsonl:   0%|          | 0.00/793M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/444655 [00:00<?, ? examples/s]


[EVAL] Downloading MATH test...
[EVAL] Downloading GSM8K test...

All downloads complete!


In [6]:
# Prepare SFT dataset
from mathphd_plus_plus.data.preprocess_sft import prepare_sft_dataset

sft_dataset = prepare_sft_dataset(
    sft_raw,
    tokenizer=tokenizer,
    config=config.sft,
    max_seq_length=config.sft.max_seq_length,
)
print(f'\nSFT dataset ready: {len(sft_dataset)} samples')
print(f'Sample: {sft_dataset[0]["text"][:200]}...')

  MetaMathQA: 39999 samples
  MATH train: 7500 samples
  GSM8K train: 7473 samples
  NuminaMath: 3000 samples

[SFT] Total before dedup: 57972
[SFT] After dedup: 39910
[SFT] After length filter (max 2048): 39903

SFT dataset ready: 39903 samples
Sample: <|im_start|>system
You are MathPhD++, an advanced mathematical reasoning assistant. Show your complete reasoning step-by-step. Verify your answer using an independent method when possible.<|im_end|>
<...


In [7]:
# Prepare GRPO dataset
from mathphd_plus_plus.data.preprocess_grpo import prepare_grpo_dataset

grpo_dataset = prepare_grpo_dataset(
    sft_raw,
    max_problems=config.grpo.problems_per_epoch,
)
print(f'\nGRPO dataset ready: {len(grpo_dataset)} verifiable problems')

[GRPO] Found 844777 verifiable problems
[GRPO] Final dataset: 500 problems
  Difficulty 1: 100 problems
  Difficulty 2: 100 problems
  Difficulty 3: 100 problems
  Difficulty 4: 100 problems
  Difficulty 5: 100 problems

GRPO dataset ready: 500 verifiable problems


In [8]:
# Prepare PRM dataset
from mathphd_plus_plus.data.preprocess_prm import prepare_prm_dataset

prm_max = 10000 if QUICK_TEST else 100000
prm_dataset = prepare_prm_dataset(prm_raw, max_samples=prm_max)
print(f'\nPRM dataset ready: {len(prm_dataset)} step examples')

[PRM] Created 10000 step-level examples
  Positive (correct steps): 10000
  Negative (incorrect steps): 0

PRM dataset ready: 10000 step examples


## 3. Stage 1: Continued Pre-Training (CPT)

Multi-objective loss: `L_total = L_NTP + 0.1 * L_structure`

**Skip this stage if short on time** — go directly to SFT for faster results.

In [9]:
SKIP_CPT = True  # Set False to run continued pre-training

if not SKIP_CPT:
    from mathphd_plus_plus.data.download import download_cpt_data
    from mathphd_plus_plus.data.preprocess_cpt import prepare_cpt_dataset
    from mathphd_plus_plus.training.cpt_trainer import run_cpt

    # Download streaming CPT data
    cpt_raw = download_cpt_data(cache_dir=CACHE_DIR)

    # Prepare
    cpt_dataset = prepare_cpt_dataset(
        cpt_raw,
        tokenizer=tokenizer,
        max_seq_length=config.cpt.max_seq_length,
        target_tokens=config.cpt.target_tokens,
    )

    # Train
    cpt_path = run_cpt(
        model=model,
        tokenizer=tokenizer,
        train_dataset=cpt_dataset,
        config=config.cpt,
        output_dir=os.path.join(GDRIVE_ROOT, 'cpt'),
    )
    print(f'CPT complete: {cpt_path}')
else:
    print('Skipping CPT stage (using base model directly for SFT)')

Skipping CPT stage (using base model directly for SFT)


## 4. Stage 2: Supervised Fine-Tuning (SFT)

ChatML format with `<thinking>`, `<answer>` structure. Response-only loss masking.

In [10]:
# from mathphd_plus_plus.training.sft_trainer import run_sft

# sft_path = run_sft(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=sft_dataset,
#     config=config.sft,
#     output_dir=os.path.join(GDRIVE_ROOT, 'sft'),
# )
# print(f'\nSFT complete: {sft_path}')

In [11]:
# Load SFT checkpoint from Google Drive (trained in prior session)
sft_path = os.path.join(GDRIVE_ROOT, 'sft/final')
if os.path.exists(sft_path):
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        sft_path,
        torch_dtype=torch.float16,
        device_map='auto',
        trust_remote_code=True,
        attn_implementation="eager",
    )
    if len(tokenizer) > model.config.vocab_size:
        model.resize_token_embeddings(len(tokenizer))
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    print(f'SFT model loaded from: {sft_path}')
    print(f'Model params: {model.num_parameters() / 1e6:.1f}M')
else:
    print(f'WARNING: SFT checkpoint not found at {sft_path}')
    print('Please run SFT training first or check the path.')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

SFT model loaded from: /content/drive/MyDrive/mathphd_checkpoints/sft/final
Model params: 494.0M


In [13]:
# Quick SFT quality check
test_problem = 'What is the sum of the first 100 positive integers?' # """Let ABC be a triangle inscribed in circle ω. Let the tangents to ω at B and C intersect at point T. Let M be the midpoint of BC. The line TM intersects the circle ω again at point D (distinct from B and C). Let E be the second intersection point of the line AD with the circle ω. Prove that TE is perpendicular to AD."""

from mathphd_plus_plus.evaluation.evaluate import generate_answer
response = generate_answer(model, tokenizer, test_problem)
print(f'Problem: {test_problem}')
print(f'Response:\n{response}')

Problem: What is the sum of the first 100 positive integers?
Response:

The sum of the first $n$ positive integers can be found using the formula $\frac{n(n+1)}{2}$.


So, for the first 100 positive integers, we have $\frac{100(100+1)}{2} = \boxed{5050}$.



## 5. Stage 3: Process Reward Model (PRM)

Trains a step-level reward model for scoring reasoning quality.

In [15]:
# from mathphd_plus_plus.training.prm_trainer import run_prm_training

prm_path = os.path.join(GDRIVE_ROOT, 'prm/final')
# prm_path = run_prm_training(
#     tokenizer=tokenizer,
#     train_dataset=prm_dataset,
#     config=config.prm,
#     output_dir=os.path.join(GDRIVE_ROOT, 'prm'),
# )
# print(f'\nPRM complete: {prm_path}')

In [16]:
# Load PRM for GRPO rewards
from mathphd_plus_plus.models.reward_model import ProcessRewardModel
from mathphd_plus_plus.rewards.process_reward import ProcessRewardScorer

prm_model = ProcessRewardModel(model_name=config.prm.model_name)
prm_state = torch.load(os.path.join(prm_path, 'prm_model.pt'), map_location='cpu')
prm_model.load_state_dict(prm_state)
prm_model = prm_model.to('cuda')

process_scorer = ProcessRewardScorer(prm_model, tokenizer, device='cuda')
print('PRM loaded for scoring')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

PRM loaded for scoring


## 6. Stage 4: GRPO Training

**The core innovation**: Group Relative Policy Optimization with:
- SymPy-verified correctness rewards
- PRM process rewards
- Format compliance rewards

This is the longest stage (~38h for full training, ~2h for quick test).

In [17]:
from mathphd_plus_plus.rewards.composite_reward import CompositeReward
from mathphd_plus_plus.training.grpo_trainer import run_grpo

gropo_path = os.path.join(GDRIVE_ROOT, 'grpo/final')
# # Create composite reward function
# reward_fn = CompositeReward(
#     process_scorer=process_scorer,
#     correctness_weight=config.grpo.reward_correctness_weight,
#     process_weight=config.grpo.reward_process_weight,
#     format_weight=config.grpo.reward_format_weight,
# )

# # Run GRPO
# grpo_path = run_grpo(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=grpo_dataset,
#     reward_fn=reward_fn,
#     config=config.grpo,
#     output_dir=os.path.join(GDRIVE_ROOT, 'grpo'),
# )
# print(f'\nGRPO complete: {grpo_path}')

## 7. Evaluation

Run benchmarks: GSM8K and MATH with difficulty and subject breakdown.

Quick-test behavior: `max_samples <= 500` uses `HuggingFaceH4/MATH-500`.
Full-run behavior: `max_samples is None` or `> 500` uses `DigitalLearningGmbH/MATH-lighteval`.

In [ ]:
from mathphd_plus_plus.evaluation.evaluate import run_all_evaluations

eval_max = 200 if QUICK_TEST else None

eval_results = run_all_evaluations(
    model=model,
    tokenizer=tokenizer,
    benchmarks=['gsm8k', 'math'],
    max_samples=eval_max,
    output_dir=os.path.join(GDRIVE_ROOT, 'eval_results'),
)

math_info = eval_results.get('math', {})
print('MATH eval dataset:', math_info.get('dataset_name', 'not loaded'))

[EVAL] Running GSM8K evaluation...


Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K: 100%|██████████| 200/200 [16:24<00:00,  4.92s/it]
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lighteval/MATH' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'lighteval/MATH' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'HuggingFaceH4/MATH-500' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like 

[GSM8K] Accuracy: 18.50% (37/200)
[EVAL] Running MATH evaluation...


...

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

[EVAL] Loaded MATH from: HuggingFaceH4/MATH-500


MATH: 100%|██████████| 200/200 [19:27<00:00,  5.84s/it]


[MATH] Overall accuracy: 6.00%

By difficulty:
  Level 3: 6.00% (12/200)

By subject:
  unknown: 6.00% (12/200)

EVALUATION SUMMARY
  gsm8k: 18.50% (37/200)
  math: 6.00% (12/200)


## 8. Inference Strategy Demos

Compare: Direct | Self-Consistency | Tree-of-Thoughts | MCTS | Multi-Agent Debate

In [19]:
from mathphd_plus_plus.inference.pipeline import InferencePipeline

pipeline = InferencePipeline(
    model=model,
    tokenizer=tokenizer,
    process_scorer=process_scorer,
    config=config.inference,
)

# Test problems of varying difficulty
test_problems = [
    {
        'problem': 'What is 15% of 240?',
        'answer': '36',
        'difficulty': 1,
    },
    {
        'problem': 'Find the remainder when 2^100 is divided by 7.',
        'answer': '2',
        'difficulty': 3,
    },
    {
        'problem': 'Let $a, b, c$ be positive reals with $a+b+c=1$. Prove that $a^2+b^2+c^2 \geq \frac{1}{3}$.',
        'answer': '1/3',
        'difficulty': 4,
    },
]

Let $a, b, c$ be positive reals with $a+b+c=1$. Prove that $a^2+b^2+c^2 \geq \frac{1}{3}$.

In [20]:
# Compare strategies
for tp in test_problems:
    print(f"\n{'='*60}")
    print(f"Problem (difficulty {tp['difficulty']}): {tp['problem'][:100]}")
    print(f"Ground truth: {tp['answer']}")
    print(f"{'='*60}")

    for strategy in ['direct', 'sc', 'tot']:
        try:
            result = pipeline.solve(
                tp['problem'],
                strategy=strategy,
                ground_truth=tp['answer'],
            )
            print(f"\n  [{strategy.upper()}] Answer: {result['answer']}")
            if 'confidence' in result:
                print(f"    Confidence: {result['confidence']:.2f}")
        except Exception as e:
            print(f"\n  [{strategy.upper()}] Error: {e}")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Problem (difficulty 1): What is 15% of 240?
Ground truth: 36

  [DIRECT] Answer: 36
    Confidence: 0.50

  [SC] Answer: 36
    Confidence: 0.94

  [TOT] Answer: 36
    Confidence: 0.50

Problem (difficulty 3): Find the remainder when 2^100 is divided by 7.
Ground truth: 2

  [DIRECT] Answer: 1
    Confidence: 0.50

  [SC] Answer: 1
    Confidence: 0.38

  [TOT] Answer: 2
    Confidence: 0.50

Problem (difficulty 4): Let $a, b, c$ be positive reals with $a+b+c=1$. Prove that $a^2+b^2+c^2 \geq \frac{1}{3}$.
Ground truth: 1/3

  [DIRECT] Answer: Equality occurs when $a=b=c=\frac{1}{3}$, so the proof is complete.
    Confidence: 0.50

  [SC] Answer: The proof is complete.
    Confidence: 0.06

  [TOT] Answer: $$\Rightarrow\left(a^2+b^2+c^2\right)\ge\frac{1}{3}$$
    Confidence: 0.50


## 9. Conjecture Generation (Bonus)

Adversarial generator-critic loop to discover novel mathematical conjectures.

In [21]:
from mathphd_plus_plus.inference.conjecture_generator import run_conjecture_generation

conjectures = run_conjecture_generation(
    model=model,
    tokenizer=tokenizer,
    num_conjectures=5,  # Small for demo
    domains=['number theory', 'combinatorics', 'linear algebra'],
)

print(f"\nInteresting (unresolved) conjectures:")
for c in conjectures.get('interesting', []):
    print(f"  - {c['conjecture'][:200]}")

print(f"\nResolved conjectures:")
for c in conjectures.get('resolved', [])[:3]:
    print(f"  - [{c['resolution']}] {c['conjecture'][:150]}")

[Conjecture Generation] Generating 5 conjectures...
  [1/5] Domain: number theory
  [2/5] Domain: combinatorics
  [3/5] Domain: linear algebra
  [4/5] Domain: number theory
  [5/5] Domain: combinatorics

[Results]
  Total: 5
  Resolved: 0
  Unresolved: 5
  Interesting (confirmed by code, unresolved by critic): 0

Interesting (unresolved) conjectures:

Resolved conjectures:


## 10. Save Final Model

Save the fully trained model to Google Drive.

In [ ]:
# Save final model
final_model_path = os.path.join(GDRIVE_ROOT, 'mathphd_final')
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f'Final model saved to: {final_model_path}')

# Optional: push to HuggingFace Hub
from huggingface_hub import login
login()
model.push_to_hub('Edmon02/mathphd-plus-plus-0.5b')
tokenizer.push_to_hub('Edmon02/mathphd-plus-plus-0.5b')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/mathphd_checkpoints/mathphd_final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp67_t15nd/tokenizer.json:  99%|#########9| 11.3MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/Edmon02/mathphd-plus-plus-0.5b/commit/eeafa3189e2a252a8a0a709193d21ae23c69ff54', commit_message='Upload tokenizer', commit_description='', oid='eeafa3189e2a252a8a0a709193d21ae23c69ff54', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Edmon02/mathphd-plus-plus-0.5b', endpoint='https://huggingface.co', repo_type='model', repo_id='Edmon02/mathphd-plus-plus-0.5b'), pr_revision=None, pr_num=None)